# RUSH Sales Analysis

####Course: GB 885  
####Author: Robin Li  


### Data acquisition

In [4]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Upload the needed files

In [7]:
# Table_products
from google.colab import files

uploaded = files.upload()


Saving TABLE_PRODUCTS_885.csv to TABLE_PRODUCTS_885 (2).csv


In [8]:
# Table_retailers
from google.colab import files

uploaded = files.upload()

Saving TABLE_RETAILER_885.csv to TABLE_RETAILER_885.csv


In [9]:
# Table_sales
from google.colab import files

uploaded = files.upload()

Saving TABLE_SALES_885.csv to TABLE_SALES_885.csv


In [10]:
# Load the three tables

df_products = pd.read_csv("TABLE_PRODUCTS_885.csv", sep="|")
df_retailer = pd.read_csv("TABLE_RETAILER_885.csv")
df_sales = pd.read_csv("TABLE_SALES_885.csv")

# Display the first few rows
print("PRODUCTS")
display(df_products.head())

print("RETAILER")
display(df_retailer.head())

print("SALES")
display(df_sales.head())

PRODUCTS


,PRODUCT_ID,PRODUCT_NAME
0,20,Men's Street Footwear
1,30,Men's Athletic Footwear
2,120,Women's Street Footwear
3,130,Women's Athletic Footwear
4,40,Men's Apparel


RETAILER


,RETAILER_ID,RETAILER,REGION,STATE,CITY
0,A00MOHCO,Amazon,Midwest,Ohio,Columbus
1,A00NMAPO,Amazon,Northeast,Maine,Portland
2,A00NMABO,Amazon,Northeast,Massachusetts,Boston
3,A00NNEMA,Amazon,Northeast,New Hampshire,Manchester
4,A00NVEBU,Amazon,Northeast,Vermont,Burlington


SALES


,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD
0,1,A00MOHCO,1/1/2020,1,1,2020,20,50.0,1200,0.5,In-store
1,7,A00MOHCO,1/7/2020,1,7,2020,20,50.0,1250,0.5,In-store
2,13,A00MOHCO,1/25/2020,1,25,2020,20,50.0,1220,0.5,Outlet
3,19,A00MOHCO,1/31/2020,1,31,2020,20,50.0,1200,0.5,Outlet
4,25,A00MOHCO,2/6/2020,2,6,2020,20,60.0,1220,0.5,Outlet


### Data Exploration


In [11]:
# Check the dimensions of each table

print("Products shape:", df_products.shape)
print("Retailer shape:", df_retailer.shape)
print("Sales shape:", df_sales.shape)

Products shape: (6, 2)
Retailer shape: (110, 5)
Sales shape: (9648, 11)


In [12]:
# Check the data type
print("PRODUCTS DATA TYPES")
print(df_products.dtypes)

print("\nRETAILER DATA TYPES")
print(df_retailer.dtypes)

print("\nSALES DATA TYPES")
print(df_sales.dtypes)

PRODUCTS DATA TYPES
PRODUCT_ID       int64
PRODUCT_NAME    object
dtype: object

RETAILER DATA TYPES
RETAILER_ID    object
RETAILER       object
REGION         object
STATE          object
CITY           object
dtype: object

SALES DATA TYPES
ORDER_ID              int64
RETAILER_ID          object
INVOICE_DATE         object
MONTH                 int64
DAY                   int64
YEAR                  int64
PRODUCT_ID            int64
PRICE_PER_UNIT      float64
UNITS_SOLD           object
OPERATING_MARGIN    float64
SALES_METHOD         object
dtype: object


In [13]:
# Check for missing values
print("Missing values in PRODUCTS:")
display(df_products.isnull().sum())

print("Missing values in RETAILER:")
display(df_retailer.isnull().sum())

print("Missing values in SALES:")
display(df_sales.isnull().sum())

Missing values in PRODUCTS:


,0
PRODUCT_ID,0
PRODUCT_NAME,0


Missing values in RETAILER:


,0
RETAILER_ID,0
RETAILER,0
REGION,0
STATE,0
CITY,0


Missing values in SALES:


,0
ORDER_ID,0
RETAILER_ID,0
INVOICE_DATE,0
MONTH,0
DAY,0
YEAR,0
PRODUCT_ID,0
PRICE_PER_UNIT,2
UNITS_SOLD,0
OPERATING_MARGIN,0


In [14]:
# Check for duplicates
print("Duplicate rows in PRODUCTS:", df_products.duplicated().sum())
print("Duplicate rows in RETAILER:", df_retailer.duplicated().sum())
print("Duplicate rows in SALES:", df_sales.duplicated().sum())

Duplicate rows in PRODUCTS: 0
Duplicate rows in RETAILER: 0
Duplicate rows in SALES: 0


### Fix errors

In [15]:
# Convert UNITS_SOLD to numeric
# Invalid values such as "***" become missing values

df_sales["UNITS_SOLD"] = pd.to_numeric(
    df_sales["UNITS_SOLD"],
    errors="coerce"
)

# Check how many values became missing
print("Missing UNITS_SOLD values:")
print(df_sales["UNITS_SOLD"].isnull().sum())

Missing UNITS_SOLD values:
2


In [16]:
# Look at descriptive statistics for price

display(df_sales["PRICE_PER_UNIT"].describe())

,PRICE_PER_UNIT
count,9646.000000
mean,55.575264
std,1017.819943
min,7.000000
25%,35.000000
50%,45.000000
75%,55.000000
max,99999.000000


In [17]:
# Find extremely high prices

display(
    df_sales[df_sales["PRICE_PER_UNIT"] > 1000]
)

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD
423,2536,F00NNEMA,5/23/2021,5,23,2021,20,99999.0,520.0,0.4,Online


### Fix the high price

In [18]:
# Replace unrealistic prices with NaN

df_sales.loc[
    df_sales["PRICE_PER_UNIT"] > 1000,
    "PRICE_PER_UNIT"
] = np.nan

In [19]:
# Check
print(df_sales["PRICE_PER_UNIT"].describe())

count    9645.000000
mean       45.213064
std        14.706062
min         7.000000
25%        35.000000
50%        45.000000
75%        55.000000
max       110.000000
Name: PRICE_PER_UNIT, dtype: float64


In [20]:
# Check sales method
print(df_sales["SALES_METHOD"].value_counts())

SALES_METHOD
Online      4889
Outlet      2999
In-store    1740
Ootlet        20
Name: count, dtype: int64


In [21]:
# Correct typo
df_sales["SALES_METHOD"] = df_sales["SALES_METHOD"].replace(
    {"Ootlet": "Outlet"}
)

# Verify
print(df_sales["SALES_METHOD"].value_counts())

SALES_METHOD
Online      4889
Outlet      3019
In-store    1740
Name: count, dtype: int64


### Create new column

In [22]:
# Calculate total sales revenue for each transaction

df_sales["SALES_DOLLARS"] = (
    df_sales["PRICE_PER_UNIT"] *
    df_sales["UNITS_SOLD"]
)

display(df_sales.head())

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD,SALES_DOLLARS
0,1,A00MOHCO,1/1/2020,1,1,2020,20,50.0,1200.0,0.5,In-store,60000.0
1,7,A00MOHCO,1/7/2020,1,7,2020,20,50.0,1250.0,0.5,In-store,62500.0
2,13,A00MOHCO,1/25/2020,1,25,2020,20,50.0,1220.0,0.5,Outlet,61000.0
3,19,A00MOHCO,1/31/2020,1,31,2020,20,50.0,1200.0,0.5,Outlet,60000.0
4,25,A00MOHCO,2/6/2020,2,6,2020,20,60.0,1220.0,0.5,Outlet,73200.0


In [23]:
# Cobine the tables
#Join sales with product information

df_analysis = df_sales.merge(
    df_products,
    on="PRODUCT_ID",
    how="left"
)

display(df_analysis.head())

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD,SALES_DOLLARS,PRODUCT_NAME
0,1,A00MOHCO,1/1/2020,1,1,2020,20,50.0,1200.0,0.5,In-store,60000.0,Men's Street Footwear
1,7,A00MOHCO,1/7/2020,1,7,2020,20,50.0,1250.0,0.5,In-store,62500.0,Men's Street Footwear
2,13,A00MOHCO,1/25/2020,1,25,2020,20,50.0,1220.0,0.5,Outlet,61000.0,Men's Street Footwear
3,19,A00MOHCO,1/31/2020,1,31,2020,20,50.0,1200.0,0.5,Outlet,60000.0,Men's Street Footwear
4,25,A00MOHCO,2/6/2020,2,6,2020,20,60.0,1220.0,0.5,Outlet,73200.0,Men's Street Footwear


In [24]:
# Join retailer information

df_analysis = df_analysis.merge(
    df_retailer,
    on="RETAILER_ID",
    how="left"
)

display(df_analysis.head())

,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD,SALES_DOLLARS,PRODUCT_NAME,RETAILER,REGION,STATE,CITY
0,1,A00MOHCO,1/1/2020,1,1,2020,20,50.0,1200.0,0.5,In-store,60000.0,Men's Street Footwear,Amazon,Midwest,Ohio,Columbus
1,7,A00MOHCO,1/7/2020,1,7,2020,20,50.0,1250.0,0.5,In-store,62500.0,Men's Street Footwear,Amazon,Midwest,Ohio,Columbus
2,13,A00MOHCO,1/25/2020,1,25,2020,20,50.0,1220.0,0.5,Outlet,61000.0,Men's Street Footwear,Amazon,Midwest,Ohio,Columbus
3,19,A00MOHCO,1/31/2020,1,31,2020,20,50.0,1200.0,0.5,Outlet,60000.0,Men's Street Footwear,Amazon,Midwest,Ohio,Columbus
4,25,A00MOHCO,2/6/2020,2,6,2020,20,60.0,1220.0,0.5,Outlet,73200.0,Men's Street Footwear,Amazon,Midwest,Ohio,Columbus


In [25]:
# Check final data
print("Final dataset shape:", df_analysis.shape)

display(df_analysis.head())

print("\nMissing values:")
display(df_analysis.isnull().sum())

Final dataset shape: (10271, 17)


,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD,SALES_DOLLARS,PRODUCT_NAME,RETAILER,REGION,STATE,CITY
0,1,A00MOHCO,1/1/2020,1,1,2020,20,50.0,1200.0,0.5,In-store,60000.0,Men's Street Footwear,Amazon,Midwest,Ohio,Columbus
1,7,A00MOHCO,1/7/2020,1,7,2020,20,50.0,1250.0,0.5,In-store,62500.0,Men's Street Footwear,Amazon,Midwest,Ohio,Columbus
2,13,A00MOHCO,1/25/2020,1,25,2020,20,50.0,1220.0,0.5,Outlet,61000.0,Men's Street Footwear,Amazon,Midwest,Ohio,Columbus
3,19,A00MOHCO,1/31/2020,1,31,2020,20,50.0,1200.0,0.5,Outlet,60000.0,Men's Street Footwear,Amazon,Midwest,Ohio,Columbus
4,25,A00MOHCO,2/6/2020,2,6,2020,20,60.0,1220.0,0.5,Outlet,73200.0,Men's Street Footwear,Amazon,Midwest,Ohio,Columbus



Missing values:


,0
ORDER_ID,0
RETAILER_ID,0
INVOICE_DATE,0
MONTH,0
DAY,0
YEAR,0
PRODUCT_ID,0
PRICE_PER_UNIT,3
UNITS_SOLD,2
OPERATING_MARGIN,0


### Business Questions

In [35]:
# Business Question 1
# Highest-selling product category in 2021

q1 = (
    df_analysis[
        (df_analysis["YEAR"] == 2021)
    ]
    .groupby("PRODUCT_NAME")["SALES_DOLLARS"]
    .sum()
    .sort_values(ascending=False)
)

print(q1)

PRODUCT_NAME
Men's Street Footwear        23275236.0
Women's Apparel              19658904.0
Men's Athletic Footwear      16702706.0
Women's Street Footwear      13807909.0
Men's Apparel                13325861.0
Women's Athletic Footwear    11381041.0
Name: SALES_DOLLARS, dtype: float64


In [34]:
# Business Question 2
# Highest women's product sales by state in 2021

q2 = (
    df_analysis[
        (df_analysis["YEAR"] == 2021) &
        (df_analysis["PRODUCT_NAME"].str.contains("Women's"))
    ]
    .groupby("STATE")["SALES_DOLLARS"]
    .sum()
    .sort_values(ascending=False)
)

print(q2.head(10))

STATE
Maine            2176301.0
Delaware         2023575.0
New Hampshire    1916400.0
Arizona          1798900.0
Missouri         1771992.0
Illinois         1743277.0
New York         1736862.0
Virginia         1719886.0
Nebraska         1712680.0
Connecticut      1600156.0
Name: SALES_DOLLARS, dtype: float64


In [33]:
# Business Question 3
# Highest men's product sales by state in 2021

q3 = (
    df_analysis[
        (df_analysis["YEAR"] == 2021) &
        (df_analysis["PRODUCT_NAME"].str.contains("Men's"))
    ]
    .groupby("STATE")["SALES_DOLLARS"]
    .sum()
    .sort_values(ascending=False)
)

print(q3.head(10))

STATE
Delaware         2334300.0
Arizona          2261025.0
Maine            2217190.0
New Hampshire    2208600.0
New York         2114999.0
Illinois         2093438.0
Missouri         1951530.0
Connecticut      1926568.0
Nebraska         1718726.0
New Mexico       1708196.0
Name: SALES_DOLLARS, dtype: float64


In [32]:
# Business Question 4a
# Retailer with the most units purchased in 2021

q4_2021 = (
    df_analysis[
        (df_analysis["YEAR"] == 2021)
    ]
    .groupby("RETAILER")["UNITS_SOLD"]
    .sum()
    .sort_values(ascending=False)
)

print(q4_2021)

RETAILER
Foot Locker      1097410.0
West Gear         315502.0
Sports Direct     256363.0
Amazon            205570.0
Kohl's            136950.0
Walmart            58286.0
Name: UNITS_SOLD, dtype: float64


In [31]:
# Business Question 4b
# Retailer with the most units purchased in 2020

q4_2020 = (
    df_analysis[
        (df_analysis["YEAR"] == 2020)
    ]
    .groupby("RETAILER")["UNITS_SOLD"]
    .sum()
    .sort_values(ascending=False)
)

print(q4_2020)

RETAILER
Amazon           317930.0
Kohl's            68686.0
West Gear         57334.0
Sports Direct     18399.0
Name: UNITS_SOLD, dtype: float64


### More Questions I came up with

In [37]:
# annual sales

annual_sales = (
    df_analysis
    .groupby("YEAR")
    .agg(
        Sales=("SALES_DOLLARS", "sum"),
        Units=("UNITS_SOLD", "sum")
    )
)

print(annual_sales)

           Sales      Units
YEAR                       
2020  24174325.0   462349.0
2021  98151657.0  2070379.0


In [39]:
# Retailer performance
retailer_sales = (
    df_analysis[df_analysis["YEAR"] == 2021]
    .groupby("RETAILER")
    .agg(
        Sales=("SALES_DOLLARS", "sum"),
        Units=("UNITS_SOLD", "sum")
    )
    .sort_values("Sales", ascending=False)
)

print(retailer_sales)

                    Sales      Units
RETAILER                            
Foot Locker    53620425.0  1097410.0
West Gear      12405097.0   315502.0
Sports Direct  12130175.0   256363.0
Amazon         10487475.0   205570.0
Kohl's          7234082.0   136950.0
Walmart         2256523.0    58286.0


In [40]:
# Which state generated the most total sales

state_sales = (
    df_analysis[df_analysis["YEAR"] == 2021]
    .groupby("STATE")["SALES_DOLLARS"]
    .sum()
    .sort_values(ascending=False)
)

print(state_sales.head(10))

STATE
Maine            4393491.0
Delaware         4357875.0
New Hampshire    4125000.0
Arizona          4059925.0
New York         3851861.0
Illinois         3836715.0
Missouri         3723522.0
Connecticut      3526724.0
Nebraska         3431406.0
Virginia         3420809.0
Name: SALES_DOLLARS, dtype: float64
